# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2

from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Drop rows with missing values (as previous lab)
spaceship_clean = spaceship.dropna().copy()

# Simplify Cabin - keep only deck letter
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]

# Drop irrelevant columns
spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])

# Define target and features
y = spaceship_clean["Transported"]
X = spaceship_clean.drop(columns=["Transported"])

# One-hot encoding for categorical features
categorical_cols = X.select_dtypes(include=["object", "bool"]).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", X_encoded.shape)

Shape after encoding: (6606, 19)


In [4]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_encoded)

# Convert to DataFrame for readability
X_scaled_df = pd.DataFrame(X_scaled, columns=X_encoded.columns)
print("Shape after scaling:", X_scaled_df.shape)

Shape after scaling: (6606, 19)


In [5]:
selector = SelectKBest(score_func=chi2, k=20)
X_selected = selector.fit_transform(X_scaled_df, y)

print("Shape after feature selection:", X_selected.shape)

Shape after feature selection: (6606, 19)


c:\Users\river\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:783: UserWarning: k=20 is greater than n_features=19. All the features will be returned.
  warnings.warn(


**Perform Train Test Split**

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (5284, 19)
X_test : (1322, 19)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [9]:
results = {}

# Bagging (bootstrap=True)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True,  
    random_state=42,
    n_jobs=-1
)

bagging.fit(X_train, y_train)
y_pred_bag = bagging.predict(X_test)
results["Bagging"] = accuracy_score(y_test, y_pred_bag)
print("Bagging accuracy:", results["Bagging"])

# Pasting (bootstrap=False)
pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=False, 
    random_state=42,
    n_jobs=-1
)

pasting.fit(X_train, y_train)
y_pred_pas = pasting.predict(X_test)
results["Pasting"] = accuracy_score(y_test, y_pred_pas)
print("Pasting accuracy:", results["Pasting"])

Bagging accuracy: 0.7753403933434191
Pasting accuracy: 0.7465960665658093


- Random Forests

In [10]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results["Random Forest"] = accuracy_score(y_test, y_pred_rf)
print("Random Forest accuracy:", results["Random Forest"])

Random Forest accuracy: 0.7791225416036308


- Gradient Boosting

In [11]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
results["Gradient Boosting"] = accuracy_score(y_test, y_pred_gb)
print("Gradient Boosting accuracy:", results["Gradient Boosting"])

Gradient Boosting accuracy: 0.7791225416036308


- Adaptive Boosting

In [12]:
ada = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)
results["AdaBoost"] = accuracy_score(y_test, y_pred_ada)
print("AdaBoost accuracy:", results["AdaBoost"])

AdaBoost accuracy: 0.7670196671709532


Which model is the best and why?

In [ ]:
# The best model was X because it achieved the highest accuracy. Ensemble methods like Random Forest or Gradient Boosting perform better because they reduce overfitting and combine multiple weak learners into a stronger, more stable model.